<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/Resnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cargar Base

In [2]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path = kagglehub.dataset_download("leonardocaravaggio/ge-images")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


100%|██████████| 13.3G/13.3G [02:35<00:00, 91.5MB/s]

Extracting files...


In [14]:
import pandas as pd
ciudades=pd.read_csv("base.csv")

In [ ]:
len(ciudades)

1095

# Resnet18

In [15]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# Cargar ResNet-18 preentrenada
resnet18 = models.resnet18(pretrained=True)
resnet18 = torch.nn.Sequential(*list(resnet18.children())[:-1])  # Quitar la capa final de clasificación
resnet18.eval()  # Poner en modo evaluación

# Transformaciones para las imágenes
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_features(img_path):
    """Extrae features de una imagen usando ResNet-18"""
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0)  # Añadir batch dimension
    with torch.no_grad():
        features = resnet18(img)
    return features.flatten().numpy()  # Convertir a vector de 512 dimensiones

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 120MB/s]


In [16]:
import numpy as np

def compute_inequality(image_1km_path, image_10km_path):
    # Extraer características de ambas imágenes
    features_1km = np.mean(extract_features(image_1km_path))
    features_10km = np.mean(extract_features(image_10km_path))

    # Calcular la diferencia entre ambas medidas de desigualdad
    difference = abs(features_1km - features_10km)

    return {
        "Desigualdad cercana (1km)": features_1km,
        "Desigualdad amplia (10km)": features_10km,
        "Diferencia entre ambas": difference
    }

In [51]:
import os

for i in range(1095):
  if pd.isna(ciudades.loc[i, "Diferencia"]):  # Solo procesar si "Diferencia" está vacío
    try:
        nombre_archivo = ciudades.City[i].replace("/",".").replace(":","_").replace("'","!")
        ruta_completa = os.path.join(path, "imagenes", nombre_archivo)

        if not os.path.exists(ruta_completa + " - 1K.png"):
          print(ruta_completa + "no existe")


        resultados = compute_inequality(ruta_completa + " - 1K.png",
                                        ruta_completa + " - 10K.png")
        ciudades.loc[i, "Desigualdad_1km"] = resultados["Desigualdad cercana (1km)"]
        ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad amplia (10km)"]
        ciudades.loc[i, "Diferencia"] = resultados["Diferencia entre ambas"]
    except Exception as e:
        print(f"⚠️ Error en compute_inequality: {e}")

# Bajar la base con el indicador de desigualdad

In [ ]:
from google.colab import files
ciudades.to_csv('base.csv')
files.download('base.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>